# 07 · 判分：尺子是怎么做的，怎么知道尺子是准的

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → [07 判分 ✅] → 08 MLflow 评测 ✅ → 09 第一个基线 → 10 起：工具、agent …
```

> **这一章的代码在合并请求 #5 里。** 第 4 步会真的跑两个对照系统，每跑一次就在 MLflow 里多一条记录，可以重复运行。

## 背景：没有尺子，"变好了"只是感觉

下一章开始要调大模型、做 agent。每做一版，都要回答"比上一版好还是差"。这需要一把自动的尺子：给它系统交的答卷和标准答案，它算出分数。

尺子有三个要求：

1. **在跑任何模型之前定好。** 先看到结果再定规则，人会不自觉地挑对自己有利的判法，比如"名字差不多就算对"
2. **自己也要被验证。** 尺子本身写错了（比如编号格式没对上，所有答案都判错），后面所有数字都是错的
3. **每个分数都能追溯、能重算。** 旧项目有一份报告写着 70%，后来重跑是 83–85%。当时没记下用的是哪版代码、哪份数据，到现在也查不清哪个是对的。这一条交给第 08 章的 MLflow

**公司里的做法**：评测框架（eval harness）做四件事：被测的系统只看得到题目；判分规则事先写死；分数带误差范围；每次评测存档、可以重算。上线之前，先用"已经知道该得多少分"的假系统校准一遍。

## 这一章做的 4 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 规定题目和答卷的格式 | `Question` / `Candidate` | 系统拿不到答案；所有系统交同一种答卷 |
| 2 | 写死"什么算答对" | 按 OMIM 编号和基因判 | 写法不同不会误判；没答的算错 |
| 3 | 分数带误差范围 | 前 k 名命中率 + Wilson 区间 | 知道两个版本的差别是不是运气 |
| 4 | 校准尺子 | 阳性对照、阴性对照 | 证明尺子是准的；知道地板在哪 |

每次评测怎么记下来（MLflow），是第 08 章的内容。

**花费：0。** 这一章还不调大模型。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 规定题目和答卷的格式

### 在干什么
规定系统**看到什么**（题目），**交回什么**（答卷）。

### 背景
以后要评测好几个系统：最简单的对照、大模型直接猜、固定流程、agent。如果每个系统交的答卷格式都不一样，判分就要写好几套。更要紧的一点：**系统绝不能看到答案。**

### 用了什么技术，为什么

| | 是什么 | 里面有什么 |
|---|---|---|
| `Question`（题目） | 系统能看到的全部 | 病例编号、症状（标准术语和名字）、性别 |
| `Candidate`（候选答案） | 答卷上的一行 | 病名、OMIM 编号、基因 |

一份答卷 = 按可能性从高到低排好的一串候选答案。

**题目里没有答案的任何一部分**：没有病名、编号、基因、来源论文。有一条测试专门检查这件事，以后谁不小心往题目里加了答案，测试就会失败。公司里叫**评测隔离**：被测的系统和标准答案隔开。

代码在 [`src/raredx/evals/systems.py`](../../src/raredx/evals/systems.py)。

### 公司里什么情况怎么做
- **接入一个新系统**：只要做到一件事：拿到题目、交回答卷（这叫**接口**，interface）。判分、存档都不用改
- **答卷格式要改**：所有系统和判分一起改，走合并请求审查

### 执行
看一道题（第 05 号那个 Perrault 病例）在系统眼里是什么样，再写一份假想的答卷：

In [ ]:
from raredx.data import cases as case_store, sources
from raredx.evals.systems import Candidate, question_of

source = sources.load_sources(ROOT / "evals/datasets.toml")["phenopacket-store"]
by_id = {c.id: c for c in case_store.read(sources.processed_dir(source, ROOT / "data"))}
case = by_id["PMID_39701103_Proband_F3"]

print("题目（系统能看到的全部）：")
print(question_of(case).model_dump_json(indent=2))

answer = [
    Candidate(disease_name="Perrault syndrome 1", disease_id="OMIM:233400", gene="HSD17B4"),
    Candidate(disease_name="Perrault syndrome 7", disease_id="621101", gene="DAP3"),
]
print("\n一份假想的答卷：")
for rank, c in enumerate(answer, 1):
    print(rank, c.model_dump())

### 结果怎么看
- 题目里只有病例编号、5 个症状、性别。病名、基因、来源论文都不在
- 答卷两行：第 1 名猜的是 Perrault 综合征 1 型（错），第 2 名是 7 型（对）。第 2 行的编号故意只写了数字 `621101`，下一步看判分认不认

### 面试怎么说
> "Every system sees only a Question — the phenotype — and returns ranked Candidates; a test guarantees the question carries no part of the answer."

---

## 第 2 步 · 写死"什么算答对"

### 在干什么
把判分规则写成代码，放在 [`src/raredx/evals/scoring.py`](../../src/raredx/evals/scoring.py) 的最上面，每条规则配一条测试。

### 背景
同一个病有好几种写法："Perrault 综合征"、"Perrault syndrome 7"、"OMIM:621101"。按文字比，对的会被判成错的；反过来，"名字差不多就算对"又会把 1 型当成 7 型。

### 用了什么技术，为什么
四条规则：

| 规则 | 为什么 |
|---|---|
| **病按 OMIM 编号判**，不比病名 | 编号是唯一的；病名写法太多 |
| **基因按基因符号判**，忽略大小写；没有致病基因的病例不算基因分 | 69 例染色体缺失的病例没有单一的致病基因 |
| **名次 = 在答卷上的位置** | 第 2 行答对，就是第 2 名 |
| **没答的算错，留在分母里** | 系统出错、超时、交白卷，都算没答对。把失败的病例丢掉，每个数字都会虚高 |

编号的写法不做要求：`OMIM:621101`、`MIM 621101`、`621101` 都认，统一成 `OMIM:621101` 再比。但**只写病名、不写编号，不算对**。所以以后给模型的要求里，会明确要它写出编号。

**为什么不让大模型当裁判**：公司里常用"让另一个大模型判断答案对不对"（LLM-as-judge）。它会判错，而且每次判的还可能不一样。我们的答案有编号，可以直接比，更可靠，也不花钱。

### 公司里什么情况怎么做
- **改规则**：改代码、改测试，走合并请求；以前的评测用新规则重新判一遍（第 5 步讲怎么做）
- **什么时候用大模型当裁判**：答案没法按编号比的时候，比如判断一段解释写得好不好。而且要先抽一批，人工核对它判得准不准

### 执行
给第 1 步那份答卷判分，再看一份只写了病名的答卷：

In [ ]:
from raredx.evals.scoring import score_case

print(score_case(case, answer))
print(score_case(case, [Candidate(disease_name="Perrault syndrome 7")]))

### 结果怎么看
- 第一份：`disease_rank=2 ... gene_rank=2`：正确的病和基因都排在第 2 名。只写了数字 `621101`，也认出来了
- 第二份：`disease_rank=None gene_rank=None`：只写病名、没写编号和基因，不算对

### 面试怎么说
> "Answers are scored by OMIM number and gene symbol rather than by name; a missing or failed answer is a miss that stays in the denominator. No LLM judge is needed because the answers carry identifiers."

---

## 第 3 步 · 分数带误差范围

### 在干什么
每个分数都报两样：**前 k 名命中率**，和它的 **95% 误差范围**。

### 背景
06 号说过，200 例的分数有大约正负 7 个百分点的误差。只报一个数，读的人会把运气当成进步。

### 用了什么技术，为什么
- **前 k 名命中率**（Recall@k，也叫 top-k accuracy）：正确答案在答卷前 k 名里的病例占多少。我们报 k = 1、3、5、10。这是这个领域最标准的指标，DeepRare 报告的 Recall@1 就是它
- **Wilson 区间**：一种算误差范围的公式，下面细讲。**你 OtoVCE 论文里用的就是它**

> **用大白话说**
>
> **分数和范围是两回事**：
>
> | | 回答什么问题 |
> |---|---|
> | 分数，比如 50% | 这个系统**有多好**：100 个病人大约答对 50 个 |
> | 范围，比如 49%–51% | 这个 50% **靠不靠得住**：换一批同样难度的题再考，分数大概会在这个范围里上下波动 |
>
> 打个比方：一个人投 10 个球进 5 个，另一个人投 6,000 个进 3,000 个，命中率都是 50%。第一个人的 50% 不靠谱，他的真实水平可能是 30%，也可能是 70%；第二个人的 50% 很靠得住。
>
> ```
> 投 10 个进 5 个          24% |============●============| 76%
> 投 6,000 个进 3,000 个                49% |●| 51%
> ```
>
> 所以**范围越窄，波动越小，分数越可信**。它就是你实验柱状图上的误差线。但要注意：范围窄只说明"准"，不说明"好"。好不好，要和地板（第 4 步的瞎猜系统）、和别的系统比。
>
> **95%** 是约定俗成的标准：有 95% 的把握，真实水平落在这个范围里。和你熟悉的 p < 0.05 是同一个"5%"。
>
> **Wilson** 是算这个范围的一个公式。为什么不用最简单的"分数加减一个数"？因为碰到接近 0% 或 100% 的分数，简单算法会算出不合理的结果。第 4 步的两个对照正好就是这种情况：
>
> | | 最简单的算法 | Wilson |
> |---|---|---|
> | 瞎猜系统前 1 名：200 例对 1 例，0.5% | −0.5% 到 1.5%：出现负数，不可能 | 0.1% 到 2.8% |
> | 作弊系统：200 例全对，100% | 100% 到 100%："绝对是 100%"，太自信 | 98.1% 到 100% |
>
> 分数在 50% 附近时，两种算法几乎一样；差别出在接近 0% 或 100%、或者题很少的时候。

### 公司里什么情况怎么做
- **比较两个版本**：看两个区间重叠多少。重叠很多，就是"分不出来"，要到更大的题集上确认。下面的执行格有一个例子
- **报告分数**：永远写清楚是在哪份题上、多少例、区间多少

### 执行
同样是答对一半，题目数不同，误差范围差多少；再看两个版本 A（50%）和 B（54%），在 200 例和 6,014 例上分不分得出来：

In [ ]:
from raredx.evals.scoring import wilson

for n in (20, 200, 6014):
    low, high = wilson(n // 2, n)
    print(f"{n:>5} 例，答对一半：50%，95% 区间 {low:.1%} – {high:.1%}")

print()
for n in (200, 6014):
    a, b = wilson(round(n * 0.50), n), wilson(round(n * 0.54), n)
    print(f"{n:>5} 例：A 50% 区间 {a[0]:.1%} – {a[1]:.1%}；B 54% 区间 {b[0]:.1%} – {b[1]:.1%}")

### 结果怎么看
- 20 例：区间约 30%–70%，几乎什么都说明不了（ci-20 只用来查"有没有坏"，不用来比谁好）
- 200 例：约 43%–57%，就是 06 号说的"正负 7 个点"
- 6,014 例：约 49%–51%
- A 和 B 差 4 个百分点：在 200 例上，A 是 43.1%–56.9%，B 是 47.1%–60.8%，两个范围大半重叠，**分不出谁好**；在 6,014 例上，A 是 48.7%–51.3%，B 是 52.7%–55.3%，完全不重叠，**B 确实更好**

### 面试怎么说
> "Every top-k rate is reported with a 95% Wilson interval, so a reader can tell a real improvement from noise."

---

## 第 4 步 · 校准尺子：阳性对照和阴性对照

### 在干什么
跑两个"事先知道该得多少分"的假系统，检查尺子准不准，顺便找出地板分。

### 背景
做实验要先跑阳性对照和阴性对照，评测也一样。尺子要是写错了，比如编号格式没对上，所有答案都会被判错，而你只会以为系统很差。

### 用了什么技术，为什么

| 对照 | 怎么答题 | 应该得多少 | 证明什么 |
|---|---|---|---|
| **阳性对照**：`oracle`（作弊系统） | 直接交标准答案 | 100% | 尺子认得出对的答案 |
| **阴性对照**：`popularity` | 不看症状，每个病人都答"论文里最常见的 10 个病" | 很低 | 地板：真正的系统至少要比它好 |

`popularity` 的两个细节：
- **按论文数算"常见"**，不按病例数。否则那篇 462 例的论文一篇就决定了答案
- **只用 dev-200 以外的病例来统计**：dev-200 不能给自己出的题打分

还有一道保护：`raredx eval run` 不允许在留出集上跑，除非加上 `--final`。**哪怕是假系统，也不在留出集上跑。**

### 公司里什么情况怎么做
- **评测框架上线前先跑对照**；每次改了评测代码，再跑一次。阳性对照不再是 100%，就是尺子坏了
- **地板要写进报告**：说"agent 前 10 名命中 40%"之前，先说"不看症状瞎猜是 9%"

### 执行

In [ ]:
run("uv run raredx eval run --system oracle --split dev-200")
run("uv run raredx eval run --system popularity --split dev-200")
run("uv run raredx eval run --system popularity --split heldout", check=False)

### 结果怎么看
- **oracle**：病和基因在每个 k 都是 `100.0%`，区间 98.1%–100%。尺子认得出对的答案
- **popularity**：前 1 名 `0.5%`（200 例里 1 例），前 10 名 `9.0%`，区间 5.8%–13.8%。这就是地板：**以后任何系统，前 10 名低于 9% 左右，就等于没看症状**
- **留出集**：`heldout is the final exam: run it once, with --final, when the design is frozen`，拒绝了
- 每次评测最后一行 `recorded in MLflow: experiment eval/dev-200, run …`：这次评测已经记进了 MLflow（第 08 章讲）。第一次运行时还会多两行 `Creating initial MLflow database tables`，那是在建 MLflow 的数据库，只有第一次会出现

### 面试怎么说
> "The harness is calibrated with a positive control — an oracle that must score 100% — and a negative control, a popularity baseline that ignores the phenotype and sets the floor at 9% top-10."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 题目和答卷 | `Question` / `Candidate` | 系统拿不到答案；所有系统一个接口 |
| 2 判分规则 | OMIM 编号、基因、没答算错 | 规则事先写死，不随结果改 |
| 3 误差范围 | 前 k 名命中率 + Wilson 区间 | 分得清进步和运气 |
| 4 校准 | 阳性对照 100%，阴性对照 9% | 尺子是准的，地板在哪清清楚楚 |

**一句话**：尺子做好了，也验证过了：作弊系统拿满分，瞎猜的系统前 10 名只有 9%。每次评测怎么记下来、怎么用 MLflow 原生的方式跑，是下一章的内容。

## 面试题

1. **How do you evaluate an agent that outputs a ranked list of diagnoses?**
   > "Systems see only the phenotype and return ranked candidates with OMIM numbers. I score by exact identifier and gene, report top-1/3/5/10 with Wilson intervals, and count failures as misses."

2. **When would you use an LLM as a judge?**
   > "Only when the answer can't be checked mechanically — say, the quality of an explanation — and only after validating the judge against human labels on a sample. Here the answers carry identifiers, so it isn't needed."

3. **How do you know your evaluation harness itself is correct?**
   > "Positive and negative controls: an oracle that must score 100%, and a trivial baseline that sets the floor. Re-run both whenever the harness changes."

**下一章 08 · 用 MLflow 原生的方式做评测**：题集登记成数据集、判分规则写成打分器、每道题一条追踪记录，评测记录可比较、可重算。